# SetFit HTS Classifier Training - Google Colab

This notebook trains a SetFit model for HTS code classification on Google Colab's free GPU.

**Before running:**
1. Runtime → Change runtime type → T4 GPU → Save
2. Upload your training data files to Colab

**Expected time:** 30-45 minutes on T4 GPU

## Step 1: Install Dependencies

In [ ]:
!pip install -q setfit==1.1.3 transformers==4.57.6 torch scikit-learn sentence-transformers datasets

## Step 2: Upload Training Data

Upload these files from your local machine:
- `crossRulings.json` (from `src/data/`)
- `crossRulings-validation.json` (from `src/data/`)
- `crossRulings-test.json` (from `src/data/`)

In [ ]:
from google.colab import files
import json

print("Upload crossRulings.json (training data)...")
uploaded = files.upload()

print("\nUpload crossRulings-validation.json...")
uploaded_val = files.upload()

print("\nUpload crossRulings-test.json...")
uploaded_test = files.upload()

## Step 3: Prepare Dataset

In [ ]:
import json
from datasets import Dataset

def load_rulings(filepath):
    with open(filepath, 'r') as f:
        data = json.load(f)
        # Handle both formats: array or object with 'rulings' key
        if isinstance(data, list):
            return data
        elif isinstance(data, dict) and 'rulings' in data:
            return data['rulings']
        else:
            raise ValueError('Unexpected data format')

def prepare_dataset(rulings, level='subheading'):
    texts = []
    labels = []
    
    for ruling in rulings:
        # Handle different field names
        text = ruling.get('productDescription') or ruling.get('product_description', '')
        hts_codes = ruling.get('htsCodes') or ruling.get('hts_codes', [])
        
        if not text or not hts_codes:
            continue
        
        # Use first HTS code
        hts_code = hts_codes[0] if isinstance(hts_codes, list) else hts_codes
        
        # Extract code level
        if level == 'chapter':
            label = hts_code[:2]
        elif level == 'heading':
            label = hts_code[:4]
        elif level == 'subheading':
            label = hts_code[:6]
        else:  # full
            label = hts_code
        
        texts.append(text)
        labels.append(label)
    
    return Dataset.from_dict({'text': texts, 'label': labels})

# Load data
print("Loading training data...")
train_rulings = load_rulings('crossRulings.json')
print(f"Loaded {len(train_rulings)} training rulings")

val_rulings = load_rulings('crossRulings-validation.json')
print(f"Loaded {len(val_rulings)} validation rulings")

test_rulings = load_rulings('crossRulings-test.json')
print(f"Loaded {len(test_rulings)} test rulings")

# Prepare datasets
print("\nPreparing datasets for subheading classification...")
train_dataset = prepare_dataset(train_rulings, level='subheading')
val_dataset = prepare_dataset(val_rulings, level='subheading')
test_dataset = prepare_dataset(test_rulings, level='subheading')

print(f"\nTraining samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Unique labels: {len(set(train_dataset['label']))}")

## Step 4: Train SetFit Model

In [ ]:
from setfit import SetFitModel, Trainer, TrainingArguments
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

# Initialize model
print("\nInitializing SetFit model...")
model = SetFitModel.from_pretrained(
    "sentence-transformers/all-MiniLM-L6-v2",
    labels=list(set(train_dataset['label']))
)

# Training arguments
args = TrainingArguments(
    batch_size=16,
    num_epochs=1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    output_dir="./setfit-hts-subheading",
)

# Create trainer
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
)

# Train!
print("\nStarting training...")
print("This will take 30-45 minutes on T4 GPU")
trainer.train()

print("\n✅ Training complete!")

## Step 5: Evaluate Model

In [ ]:
import time
from sklearn.metrics import accuracy_score, classification_report

print("Evaluating on test set...")
start_time = time.time()

# Make predictions
predictions = model.predict(test_dataset['text'])

# Calculate metrics
accuracy = accuracy_score(test_dataset['label'], predictions)
inference_time = (time.time() - start_time) / len(test_dataset) * 1000  # ms per sample

print(f"\n{'='*60}")
print(f"Test Accuracy: {accuracy*100:.2f}%")
print(f"Avg Inference Time: {inference_time:.2f}ms per sample")
print(f"{'='*60}")

# Save metrics
metrics = {
    'accuracy': accuracy,
    'total_samples': len(test_dataset),
    'inference_time_ms': inference_time,
    'model_name': 'setfit-hts-subheading'
}

with open('./setfit-hts-subheading/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("\nMetrics saved to metrics.json")

## Step 6: Test Sample Predictions

In [ ]:
# Test on some examples
test_examples = [
    "cotton t-shirt",
    "plastic water bottle",
    "leather wallet",
    "stainless steel kitchen knife",
    "wooden dining table"
]

print("Sample predictions:\n")
for example in test_examples:
    pred = model.predict([example])[0]
    print(f"'{example}' → {pred}")

## Step 7: Save and Download Model

In [ ]:
# Save model
print("Saving model...")
model.save_pretrained('./setfit-hts-subheading')
print("Model saved to ./setfit-hts-subheading/")

# Create a zip file for download
!zip -r setfit-hts-subheading.zip setfit-hts-subheading/

print("\nDownloading model...")
files.download('setfit-hts-subheading.zip')

print("\n✅ Done! Extract the zip file and place it in your models/ directory.")

## Next Steps

1. Download the `setfit-hts-subheading.zip` file
2. Extract it to your project: `models/setfit-hts-subheading/`
3. Restart your inference server:
   ```bash
   python scripts/setfit-inference-server.py \
     --model models/setfit-hts-subheading \
     --port 5001
   ```
4. Test your classification API - it should now be 90%+ accurate!